# 12 — Interactive Lab (embed existing GUIs)

**This notebook iframes the existing GUIs.** It is not a widget redesign.

The kernel name **CXR local Qwen (faiss_gpu1)** is only the Python env. It does **not** mean Qwen is loaded. Do **not** `boot()` / `from_pretrained` here. The GUIs must display with no model in this kernel.

Do **not** click Live GPU on the embedded GUIs yet (that still loads a private copy). Shared Live is `:8270`, later.

Open Jupyter at **http://127.0.0.1:8266**. Run **Start servers**, then the **embed** cells.



## 0. Start the existing GUI servers (no Qwen load)


In [9]:
import subprocess, time, urllib.request
from pathlib import Path

PY = "/home/udonsi-kalu/staging/cxrlabs/faiss_gpu1/bin/python"
WB = Path("/home/udonsi-kalu/staging/cxr-n2s-eval-workbench")
LAB = Path("/home/udonsi-kalu/staging/cxr-evidence-grounding-lab")
LOG = Path("/tmp/cxr-gui-ports")
LOG.mkdir(exist_ok=True)

SERVERS = [
    (8257, WB, "server.py"),
    (8258, WB, "upstream_server.py"),
    (8259, LAB, "nn_layer_loss_server.py"),
]

def up(port, timeout=1.0):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=timeout)
        return True
    except Exception:
        return False

for port, cwd, script in SERVERS:
    if up(port):
        print(f":{port} already up")
        continue
    log = LOG / f"{port}.log"
    proc = subprocess.Popen(
        [PY, script],
        cwd=str(cwd),
        stdout=open(log, "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print(f":{port} starting pid={proc.pid}")
    for _ in range(30):
        time.sleep(0.3)
        if up(port):
            print(f":{port} HTTP 200")
            break
    else:
        print(f":{port} not answering — see {log}")

print("---")
for port, _, _ in SERVERS:
    print(f":{port}", "UP" if up(port, 2) else "DOWN")


:8257 already up
:8258 already up
:8259 already up
---
:8257 UP
:8258 UP
:8259 UP


## 0b. Is an LLM running?

The iframe does **not** print into the notebook. On `:8257` look for the small gray line under the buttons: `Evaluating… (Ollama extract + Dual)`. Easy to miss.

Run the next cell, then click **Evaluate**. This poll is the notebook’s LLM/GPU indicator.


In [2]:
import json, subprocess, time, urllib.request
from IPython.display import clear_output, display, Markdown

def _nvidia():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-compute-apps=pid,used_gpu_memory,process_name",
             "--format=csv,noheader"],
            text=True, timeout=3,
        ).strip()
    except Exception as e:
        return f"nvidia-smi failed: {e}"
    return out or "(no compute apps)"

def _ollama():
    try:
        out = subprocess.check_output(["ollama", "ps"], text=True, timeout=3).strip()
    except Exception:
        try:
            out = subprocess.check_output(["/usr/local/bin/ollama", "ps"], text=True, timeout=3).strip()
        except Exception as e:
            return f"ollama ps failed: {e}"
    lines = [ln for ln in out.splitlines() if ln.strip()]
    if len(lines) <= 1:
        return "Ollama: idle (no runner)"
    return out

def _8270():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8270/status", timeout=0.8) as r:
            d = json.loads(r.read().decode())
        if d.get("loaded"):
            return f":8270 HF Qwen LOADED  pid={d.get('pid')}  {d.get('model_id')}  alloc={d.get('cuda_alloc_GiB')} GiB"
        return f":8270 up, weights not loaded  pid={d.get('pid')}"
    except Exception:
        return ":8270 down"

def snapshot():
    ollama = _ollama()
    running = "idle (no runner)" not in ollama
    tag = "**OLLAMA IS RUNNING**" if running else "Ollama idle"
    return (
        f"### LLM / GPU  `{time.strftime('%H:%M:%S')}`\n"
        f"{tag}\n\n"
        f"```\n{ollama}\n```\n"
        f"**HF service**\n\n`{_8270()}`\n\n"
        f"**nvidia-smi compute**\n\n```\n{_nvidia()}\n```\n"
        "Ollama = `:8257` Evaluate. HF `:8270` = resident 7B. "
        "A new ~14 GiB python pid besides `:8270` = a GUI Live/Intervene private load."
    )

def watch_llms(seconds=90, every=2.0):
    """Run this, then click Evaluate. Ctrl+C / interrupt kernel to stop early."""
    t0 = time.time()
    while time.time() - t0 < seconds:
        clear_output(wait=True)
        display(Markdown(snapshot()))
        time.sleep(every)
    clear_output(wait=True)
    display(Markdown(snapshot() + "\n\n_watch ended_"))

display(Markdown(snapshot()))
print("watch_llms(90)  → run that, then click Evaluate in :8257")


### LLM / GPU  `06:44:01`
Ollama idle

```
Ollama: idle (no runner)
```
**HF service**

`:8270 HF Qwen LOADED  pid=462788  Qwen/Qwen2.5-7B-Instruct  alloc=14.19 GiB`

**nvidia-smi compute**

```
462788, 14954 MiB, /home/udonsi-kalu/staging/cxrlabs/faiss_gpu1/bin/python
446707, 256 MiB, /home/udonsi-kalu/staging/cxrlabs/faiss_gpu1/bin/python
```
Ollama = `:8257` Evaluate. HF `:8270` = resident 7B. A new ~14 GiB python pid besides `:8270` = a GUI Live/Intervene private load.

watch_llms(90)  → run that, then click Evaluate in :8257


## 1. Embed helper


In [3]:
from IPython.display import IFrame, HTML, display

def embed_gui(port: int, title: str, height: int = 780):
    url = f"http://127.0.0.1:{port}/"
    display(HTML(
        f'<p style="margin:0 0 8px 0;"><strong>{title}</strong> · '
        f'<a href="{url}" target="_blank">{url}</a> '
        f"(open this if the frame is blank)</p>"
    ))
    display(IFrame(src=url, width="100%", height=height))

print("embed_gui ready")
print("iframe src = http://127.0.0.1:<port>/  (needs JupyterLab allowedIframeHosts)")


embed_gui ready
iframe src = http://127.0.0.1:<port>/  (needs JupyterLab allowedIframeHosts)


## 2. GUI `:8257` — eval workbench


In [4]:
embed_gui(8257, "8257 — eval workbench")


## 3. GUI `:8258` — upstream A/B score


In [5]:
embed_gui(8258, "8258 — upstream A/B score")


## 4. GUI `:8259` — nn layer-loss


In [6]:
embed_gui(8259, "8259 — nn layer-loss")


## Claim boundary

✓ Existing GUIs, started if needed, shown in iframes.  
✗ Does not load Qwen. Do not click Live GPU on these frames yet.  
✗ Jupyter kernel `model` is not shared with these HTTP processes.
